In [1]:
from langchain_community.document_loaders import DirectoryLoader

path = "data" 
loader = DirectoryLoader(path, glob="**/*.md")
docs = loader.load()

libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.


KG Construction

In [2]:
from ragas.testset.graph import KnowledgeGraph
from ragas.testset.graph import Node, NodeType


kg = KnowledgeGraph()
for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={
                "page_content": doc.page_content,
                "document_metadata": doc.metadata,
            },
        )
    )

kg

c:\Users\Aoxiong\miniconda3\envs\RASAS\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KnowledgeGraph(nodes: 8, relationships: 0)

Set up the LLM and Embedding Model

In [3]:
from ragas.llms.base import llm_factory
from ragas.embeddings import OpenAIEmbeddings
import openai

llm = llm_factory(model= 'gpt-4o-mini')
openai_client = openai.OpenAI()
embedding = OpenAIEmbeddings(client=openai_client)

Setup Extractors and Relationship builders

In [4]:
from ragas.testset.persona import Persona

person= Persona(
    name="Frontline FFF Printer Operator",
    role_description=(
        "I am a frontline FFF printer operator who asks short, practical questions I can execute immediately and safely on the shop floor. "
        "I focus on routine setup, slicer tuning, and quick troubleshooting for issues like stringing/oozing, hull lines on benchmark parts, and layer shifting across common Prusa-class machines. "
        "Answers should be concise, grounded in the provided docs, and oriented toward clear next actions and basic maintenance checks."
    ),
)

persona_list = [person]

persona_list

[Persona(name='Frontline FFF Printer Operator', role_description='I am a frontline FFF printer operator who asks short, practical questions I can execute immediately and safely on the shop floor. I focus on routine setup, slicer tuning, and quick troubleshooting for issues like stringing/oozing, hull lines on benchmark parts, and layer shifting across common Prusa-class machines. Answers should be concise, grounded in the provided docs, and oriented toward clear next actions and basic maintenance checks.')]

Single-hop Generation

In [5]:
from ragas.testset.transforms import apply_transforms
from ragas.testset.transforms import (
    HeadlinesExtractor,
    HeadlineSplitter,
    KeyphrasesExtractor,
)
from copy import deepcopy

kg_single = deepcopy(kg)

headline_extractor = HeadlinesExtractor(llm=llm)
headline_splitter = HeadlineSplitter(min_tokens=4, max_tokens=32)
keyphrase_extractor = KeyphrasesExtractor(
    llm=llm, property_name="keyphrases", max_num=10
)

transforms = [
    headline_extractor,
    headline_splitter,
    keyphrase_extractor,
]

apply_transforms(kg_single, transforms=transforms)

Applying KeyphrasesExtractor: 100%|██████████| 437/437 [09:19<00:00,  1.28s/it]


In [6]:
from ragas.testset.synthesizers.single_hop import (
    SingleHopQuerySynthesizer,
    SingleHopScenario,
)
from dataclasses import dataclass
from ragas.testset.synthesizers.prompts import (
    ThemesPersonasInput,
    ThemesPersonasMatchingPrompt,
)


@dataclass
class MySingleHopScenario(SingleHopQuerySynthesizer):

    theme_persona_matching_prompt = ThemesPersonasMatchingPrompt()

    async def _generate_scenarios(self, n, knowledge_graph, persona_list, callbacks):

        property_name = "keyphrases"
        nodes = []
        for node in knowledge_graph.nodes:
            if node.type.name == "CHUNK" and node.get_property(property_name):
                nodes.append(node)

        number_of_samples_per_node = max(1, n // len(nodes))

        scenarios = []
        for node in nodes:
            if len(scenarios) >= n:
                break
            themes = node.properties.get(property_name, [""])
            prompt_input = ThemesPersonasInput(themes=themes, personas=persona_list)
            persona_concepts = await self.theme_persona_matching_prompt.generate(
                data=prompt_input, llm=self.llm, callbacks=callbacks
            )
            base_scenarios = self.prepare_combinations(
                node,
                themes,
                personas=persona_list,
                persona_concepts=persona_concepts.mapping,
            )
            scenarios.extend(
                self.sample_combinations(base_scenarios, number_of_samples_per_node)
            )

        return scenarios

query_single = MySingleHopScenario(llm=llm)


In [7]:
scenarios = await query_single.generate_scenarios(
    n=20, knowledge_graph=kg_single, persona_list=persona_list
)

scenarios[0]

SingleHopScenario(
nodes=1
term=3D
persona=name='Frontline FFF Printer Operator' role_description='I am a frontline FFF printer operator who asks short, practical questions I can execute immediately and safely on the shop floor. I focus on routine setup, slicer tuning, and quick troubleshooting for issues like stringing/oozing, hull lines on benchmark parts, and layer shifting across common Prusa-class machines. Answers should be concise, grounded in the provided docs, and oriented toward clear next actions and basic maintenance checks.'
style=Poor grammar
length=medium)

In [8]:
import pandas as pd

rows = []
for s in scenarios:                       
    r = await query_single.generate_sample(s)
    rows.append({
        "user_input": r.user_input,
        "reference": getattr(r, "reference", None),
        "context": getattr(r, "context", None),
        "answer": getattr(r, "answer", None),
        "scenario_id": getattr(r, "scenario_id", None),
        "persona_name": getattr(r, "persona_name", None),
    })
df_single = pd.DataFrame(rows)

df_single.head(5)

,user_input,reference,context,answer,scenario_id,persona_name
0,What is extruder blob in 3D printing?,Extruder blob is a mass of plastic accumulated...,None,None,None,None
1,how to remove extruder blob if its impossible?,"Removing an extruder blob is quite tricky, tho...",None,None,None,None
2,What are the video features related to the Ori...,"The video features our Original Prusa MK3, foc...",None,None,None,None
3,How do I preheat the nozzle on FDM printer mod...,"To preheat the nozzle on FDM printer models, g...",None,None,None,None
4,How hot I need to preheat PLA for blob fix?,"For PLA, you need to preheat to 250°C to fix t...",None,None,None,None


Muti-hop Generation

In [9]:
from ragas.testset.transforms import Parallel, apply_transforms
from ragas.testset.transforms import (
    HeadlinesExtractor,
    HeadlineSplitter,
    KeyphrasesExtractor,
    OverlapScoreBuilder,
)

kg_multi = deepcopy(kg)

headline_extractor = HeadlinesExtractor(llm=llm)
headline_splitter = HeadlineSplitter(min_tokens=300, max_tokens=1000)
keyphrase_extractor = KeyphrasesExtractor(
    llm=llm, property_name="keyphrases", max_num=10
)
relation_builder = OverlapScoreBuilder(
    property_name="keyphrases",
    new_property_name="overlap_score",
    threshold=0.01,
    distance_threshold=0.9,
)


transforms = [
    headline_extractor,
    headline_splitter,
    keyphrase_extractor,
    relation_builder,
]

apply_transforms(kg_multi, transforms=transforms)

Applying OverlapScoreBuilder: 100%|██████████| 1/1 [00:00<00:00, 22.11it/s]


In [12]:
from dataclasses import dataclass
import typing as t
from ragas.testset.synthesizers.multi_hop.base import (
    MultiHopQuerySynthesizer,
    MultiHopScenario,
)
from ragas.testset.synthesizers.prompts import (
    ThemesPersonasInput,
    ThemesPersonasMatchingPrompt,
)


@dataclass
class MyMultiHopQuery(MultiHopQuerySynthesizer):

    theme_persona_matching_prompt = ThemesPersonasMatchingPrompt()

    async def _generate_scenarios(
        self,
        n: int,
        knowledge_graph,
        persona_list,
        callbacks,
    ) -> t.List[MultiHopScenario]:

        # query and get (node_a, rel, node_b) to create multi-hop queries
        results = kg_multi.find_two_nodes_single_rel(
            relationship_condition=lambda rel: (
                True if rel.type == "keyphrases_overlap" else False
            )
        )

        num_sample_per_triplet = max(1, n // len(results))

        scenarios = []
        for triplet in results:
            if len(scenarios) < n:
                node_a, node_b = triplet[0], triplet[-1]
                overlapped_keywords = triplet[1].properties["overlapped_items"]
                if overlapped_keywords:

                    # match the keyword with a persona for query creation
                    themes = list(dict(overlapped_keywords).keys())
                    prompt_input = ThemesPersonasInput(
                        themes=themes, personas=persona_list
                    )
                    persona_concepts = (
                        await self.theme_persona_matching_prompt.generate(
                            data=prompt_input, llm=self.llm, callbacks=callbacks
                        )
                    )

                    overlapped_keywords = [list(item) for item in overlapped_keywords]

                    # prepare and sample possible combinations
                    base_scenarios = self.prepare_combinations(
                        [node_a, node_b],
                        overlapped_keywords,
                        personas=persona_list,
                        persona_item_mapping=persona_concepts.mapping,
                        property_name="keyphrases",
                    )

                    # get number of required samples from this triplet
                    base_scenarios = self.sample_diverse_combinations(
                        base_scenarios, num_sample_per_triplet
                    )

                    scenarios.extend(base_scenarios)

        return scenarios

query_multi = MyMultiHopQuery(llm=llm)
scenarios_multi = await query_multi.generate_scenarios(
    n=20, knowledge_graph=kg_multi, persona_list=persona_list
)

scenarios_multi[4]

MultiHopScenario(
nodes=2
combinations=['Original Prusa MK3', 'Original Prusa MK-series']
style=Perfect grammar
length=short
persona=name='Frontline FFF Printer Operator' role_description='I am a frontline FFF printer operator who asks short, practical questions I can execute immediately and safely on the shop floor. I focus on routine setup, slicer tuning, and quick troubleshooting for issues like stringing/oozing, hull lines on benchmark parts, and layer shifting across common Prusa-class machines. Answers should be concise, grounded in the provided docs, and oriented toward clear next actions and basic maintenance checks.')

In [13]:
rows = []
for s in scenarios_multi:                       
    r = await query_multi.generate_sample(s)
    rows.append({
        "user_input": r.user_input,
        "reference": getattr(r, "reference", None),
        "context": getattr(r, "context", None),
        "answer": getattr(r, "answer", None),
        "scenario_id": getattr(r, "scenario_id", None),
        "persona_name": getattr(r, "persona_name", None),
    })
df_multi= pd.DataFrame(rows)

df_multi.head(5)

,user_input,reference,context,answer,scenario_id,persona_name
0,Why does the Benchy hull line appear on prints...,The Benchy hull line appears on prints of the ...,None,None,None,None
1,Wht are the common first layer issues and how ...,Common first layer issues include poor adhesio...,None,None,None,None
2,How can I troubleshoot a bed preheat error by ...,"To troubleshoot a bed preheat error, first ens...",None,None,None,None
3,What should I check if I encounter a Preheat e...,If you encounter a Preheat error related to th...,None,None,None,None
4,What steps should I take to check the belt ten...,To check the belt tension on your Original Pru...,None,None,None,None


Merge Datasets

In [14]:
df = pd.concat([df_single, df_multi], ignore_index=True)
df.to_csv("evaluate_dataset/test_dataset.csv",index=False)